In [1]:
import os
import json
import pandas as pd
from tqdm import tqdm



from utils.prompts import gen_qa_prompt
from utils.bedrock_functions import invoke_bedrock_endpoint, build_anthropic_request_body, build_mistral_request_body
from utils.bedrock_functions import build_command_r_request_body, build_llama_request_body, build_nova_request_body

from utils.subgraph_functions import plot_graph_with_simplified_labels, prune_triples, restore_full_triples
from utils.helper_functions import read_jsonl_file, save_as_jsonl



In [2]:
split = 'train'
directory = f"/home/ec2-user/preetam_experiments/outputs/{split}"

df = pd.read_json(f"{directory}/cleaned_subgraph_df_{split}.json", orient="records")

FileNotFoundError: File /home/ec2-user/preetam_experiments/outputs/train/cleaned_subgraph_df_train.json does not exist

In [ ]:
df

,subgraph_Steiner,subgraph_Steiner_length,subgraph_Steiner_largest_connected,subgraph_Steiner_largest_connected_length,QID,unwanted_filter_flag,unwanted_percentage
0,[[http://yago-knowledge.org/resource/Ádám_Bogd...,11,[[http://yago-knowledge.org/resource/Ádám_Bogd...,11,Q6166625,False,0.000000
1,"[[http://yago-knowledge.org/resource/Plant, ht...",16,"[[http://yago-knowledge.org/resource/Plant, ht...",12,Q1165197,False,0.000000
2,[[http://yago-knowledge.org/resource/Bade_Miya...,13,[[http://yago-knowledge.org/resource/Bade_Miya...,13,Q7916535,False,0.000000
3,[[http://yago-knowledge.org/resource/The_Sixth...,16,[[http://yago-knowledge.org/resource/The_Sixth...,14,Q28746328,False,0.000000
4,[[http://yago-knowledge.org/resource/Sean_McGi...,14,[[http://yago-knowledge.org/resource/Sean_McGi...,14,Q1436168,False,0.000000
...,...,...,...,...,...,...,...
19995,[[http://yago-knowledge.org/resource/William_T...,77,[[http://yago-knowledge.org/resource/William_T...,77,Q1379980,True,45.454545
19996,[[http://yago-knowledge.org/resource/Zhong_Che...,13,[[http://yago-knowledge.org/resource/Zhong_Che...,11,Q60763312,True,45.454545
19997,[[http://yago-knowledge.org/resource/Tennessee...,15,[[http://yago-knowledge.org/resource/Tennessee...,11,Q4916833,True,45.454545
19998,[[http://yago-knowledge.org/resource/Ben_Morri...,11,[[http://yago-knowledge.org/resource/Ben_Morri...,11,Q7988455,True,45.454545


In [4]:
str(df.iloc[0]['QID']).zfill(11)

'000Q6166625'

In [5]:
llm_requests_list = []

for i in tqdm(range(len(df))):
    recordId = str(df.iloc[i]['QID']).zfill(11)
    readable_triples = prune_triples(df.iloc[i]['subgraph_Steiner_largest_connected'])
    prompt = gen_qa_prompt(readable_triples)
    
    request_json = build_anthropic_request_body(
            system_prompt="You are helpful assistant.",
            user_prompt=prompt,
            max_tokens=2048,
            temperature=0
        )
    
    request_entry = {}
    request_entry['recordId'] = recordId
    request_entry['modelInput'] = request_json
    
    llm_requests_list.append(request_entry)

100%|██████████| 20000/20000 [00:03<00:00, 5625.44it/s]


In [7]:
save_as_jsonl(llm_requests_list, f"/home/ec2-user/preetam_experiments/batch_processing_files/{split}/llm_requests_{split}.jsonl") 


In [6]:
f"/home/ec2-user/preetam_experiments/batch_processing_files/{split}/llm_requests_{split}.jsonl"

'/home/ec2-user/preetam_experiments/batch_processing_files/validation/llm_requests_validation.jsonl'

In [8]:
llm_requests_list[32]

{'recordId': '000Q1969361',
 'modelInput': {'anthropic_version': 'bedrock-2023-05-31',
  'system': 'You are helpful assistant.',
  'messages': [{'role': 'user',
    'content': "You are an AI assistant tasked with generating question-answer pairs from knowledge graph triples. Your goal is to create natural, human-like questions and their corresponding answers based on the provided graph data.\n\nTask Overview:\nGenerate **multi-hop, complex Q&A pairs** where the questions appear simple and natural but require reasoning across multiple connected relationships within the graph to infer the answer.\n\nGuidelines for Generating Q&A Pairs:\n1. **Question Design**:\n- Questions should utilize multiple connected relationships in the graph, requiring multi-hop reasoning.\n- Avoid single-hop or trivial questions directly derived from a single triple.\n- The answer should be an entity or node in the graph.\n\n2. **Multi-Hop Reasoning**:\n- Use paths connecting entities indirectly through multiple

In [6]:
prompt = gen_qa_prompt(readable_triples)

In [14]:
def build_mistral_request_body(prompt: str, max_tokens: int = 50, temperature: float = 0.7) -> dict:
    """
    Builds a minimal JSON payload for Mistral.

    :param prompt: The input text for the model.
    :param max_tokens: The maximum number of tokens to generate (default: 50).
    :param temperature: Sampling temperature for response variation (default: 0.7).
    :return: A dict representing the minimal request body.
    """
    request_body = {
        "modelId": "mistral.mistral-small-2402-v1:0",
        "contentType": "application/json",
        "accept": "application/json",
        "body": {
            "prompt": f"<s>[INST] {prompt} [/INST]",
            "max_tokens": max_tokens,
            "temperature": temperature
        }
    }
    return request_body

In [7]:
prompt = '''
As an expert evaluator, your role is to assess the quality and validity of trivia or natural questions. These questions aim to test the responder's knowledge, which may require implicit or external information. Your goal is to analyze the question based on the following criteria:

- **Logical Structure**: Verify if the grammar and syntax are correct. (True if grammatically and syntactically correct; False if there are issues with grammar or syntax.)
- **Redundancy**: Confirm that the question does not contain its own answer explicitly or through overly obvious phrasing. (True if it contains its answer; False if it does not.)
- **Multiple Answers**: Determine if the question allows for multiple valid answers. This is acceptable in some cases, but flag it if it reduces the question's effectiveness or specificity. (True if multiple answers are plausible; False if only one valid answer is expected.)
    
#### Output JSON Keys:
- `question`: The input question.
- `logical_structure_flag`: (True/False)
- `logical_structure_reasoning`: Reason for the logical structure flag.
- `redundancy_flag`: (True/False)
- `redundancy_reasoning`: Reason for the redundancy flag.
- `multiple_answers_flag`: (True/False)
- `multiple_answers_reasoning`: Reason for the multiple answers flag.

#### Task:
Analyze the following question and provide a JSON object containing flags and reasons for potential issues:

**Question**: "Which player has played for both Melbourne Victory FC and Brentford F.C.?"

#### Output:
Return a JSON object that evaluates the question based on the criteria above.
'''

In [8]:
import boto3
import json
bedrock_runtime = boto3.client(service_name="bedrock-runtime", region_name="us-east-1")

INFO:botocore.credentials:Found credentials in shared credentials file: ~/.aws/credentials


In [9]:


def test_bedrock_pipeline():
    # Initialize the Bedrock runtime client
    bedrock_runtime = boto3.client(service_name="bedrock-runtime", region_name="us-east-1")

    # Minimal request body
    request_body = {
        "modelId": "mistral.mistral-small-2402-v1:0",
        "contentType": "application/json",
        "accept": "application/json",
        "body": json.dumps({
            "prompt": f"<s>[INST] {prompt} [/INST]",
            "max_tokens": 2048,  # Minimal token count for quick test
            "temperature": 0
        })
    }

    # Invoke the endpoint
    response = bedrock_runtime.invoke_model(
        body=request_body["body"],
        modelId=request_body["modelId"],
        contentType=request_body["contentType"]
    )

    # Decode and print the response
    response_body = json.loads(response["body"].read().decode("utf-8"))
    print(json.dumps(response_body, indent=2))
    return response_body

# Run the test
response_mistral = test_bedrock_pipeline()


{
  "outputs": [
    {
      "text": " {\n  \"question\": \"Which player has played for both Melbourne Victory FC and Brentford F.C.?\",\n  \"logical_structure_flag\": true,\n  \"logical_structure_reasoning\": \"The question is grammatically and syntactically correct.\",\n  \"redundancy_flag\": false,\n  \"redundancy_reasoning\": \"The question does not contain its own answer explicitly or through overly obvious phrasing.\",\n  \"multiple_answers_flag\": true,\n  \"multiple_answers_reasoning\": \"Multiple players could have played for both Melbourne Victory FC and Brentford F.C., depending on the time period considered.\"\n}",
      "stop_reason": "stop"
    }
  ]
}


In [5]:
json.loads(response_mistral['outputs'][0]['text'])

{'question': 'Which player has played for both Melbourne Victory FC and Brentford F.C.?',
 'logical_structure_flag': True,
 'logical_structure_reasoning': 'The question is grammatically and syntactically correct.',
 'redundancy_flag': False,
 'redundancy_reasoning': 'The question does not contain its own answer explicitly or through overly obvious phrasing.',
 'multiple_answers_flag': True,
 'multiple_answers_reasoning': 'Multiple players could have played for both Melbourne Victory FC and Brentford F.C., depending on the time period considered.'}

In [8]:
request_nova = build_nova_request_body(prompt=prompt, max_tokens=2048, temperature=0)

# Invoke the endpoint
response = bedrock_runtime.invoke_model(
    body=request_nova["body"],
    modelId=request_nova["modelId"],
    contentType=request_nova["contentType"]
)

# Decode and print the response
request_nova = json.loads(response["body"].read().decode("utf-8"))
print(json.dumps(request_nova, indent=2))

TypeError: string indices must be integers

In [9]:
request_nova

{'output': {'message': {'content': [{'text': '```json\n{\n  "question": "Which player has played for both Melbourne Victory FC and Brentford F.C.?",\n  "logical_structure_flag": true,\n  "logical_structure_reasoning": "The question is grammatically and syntactically correct.",\n  "redundancy_flag": false,\n  "redundancy_reasoning": "The question does not contain its own answer explicitly or through overly obvious phrasing.",\n  "multiple_answers_flag": true,\n  "multiple_answers_reasoning": "There could be multiple players who have played for both Melbourne Victory FC and Brentford F.C., making it possible for more than one valid answer to exist."\n}\n```'}],
   'role': 'assistant'}},
 'stopReason': 'end_turn',
 'usage': {'inputTokens': 339, 'outputTokens': 143, 'totalTokens': 482}}

In [14]:
import json

def build_nova_request_body(
    prompt: str,
    max_tokens: int = 2048,
    temperature: float = 0
) -> dict:
    """
    Builds a minimal JSON payload for Amazon Nova for single-prompt inference.

    :param prompt: The input text for the model.
    :param max_new_tokens: The maximum number of tokens to generate in the response (default: 1000).
    :param temperature: Sampling temperature for response variation (default: 0.7).
    :return: A dict representing the request body, with the 'body' serialized as a JSON string.
    """
    # Build the request body
    request_body = {
        "modelId": "amazon.nova-lite-v1:0",
        "contentType": "application/json",
        "accept": "application/json",
        "body": json.dumps({  # Serialize the body as JSON string
            "inferenceConfig": {
                "max_new_tokens": max_tokens,
                "temperature": temperature
            },
            "messages": [
                {
                    "role": "user",
                    "content": [
                        {
                            "text": prompt
                        }
                    ]
                }
            ]
        })
    }
    return request_body


In [10]:
bedrock_runtime = boto3.client(service_name="bedrock-runtime", region_name="us-east-1")

request_commandr = build_command_r_request_body(prompt=prompt, max_tokens=2048, temperature=0)

# Invoke the endpoint
response = bedrock_runtime.invoke_model(
    body=request_commandr["body"],
    modelId=request_commandr["modelId"],
    contentType=request_commandr["contentType"]
)

# Decode and print the response
response_commandr = json.loads(response["body"].read().decode("utf-8"))
print(json.dumps(response_commandr, indent=2))


{
  "response_id": "379ed018/020835bc-acff-4663-9d4a-75f3811124d1",
  "text": "```json\n{\n  \"question\": \"Which player has played for both Melbourne Victory FC and Brentford F.C.?\",\n  \"logical_structure_flag\": true,\n  \"logical_structure_reasoning\": \"The question is grammatically and syntactically correct.\",\n  \"redundancy_flag\": false,\n  \"redundancy_reasoning\": \"The question does not contain any explicit answers or obvious phrasing that might lead to confusion.\",\n  \"multiple_answers_flag\": true,\n  \"multiple_answers_reasoning\": \"The question could potentially have multiple valid answers, as more than one player might fit the description.\"\n}\n```",
  "generation_id": "b9a4db57-f882-46ea-acd3-a610698470bc",
  "chat_history": [
    {
      "role": "USER",
      "message": "\nAs an expert evaluator, your role is to assess the quality and validity of trivia or natural questions. These questions aim to test the responder's knowledge, which may require implicit or e

In [11]:

request_llama = build_llama_request_body(prompt=prompt, max_tokens=2048, temperature=0)

# Invoke the endpoint
response = bedrock_runtime.invoke_model(
    body=request_llama["body"],
    modelId=request_llama["modelId"],
    contentType=request_llama["contentType"]
)

# Decode and print the response
response_llama = json.loads(response["body"].read().decode("utf-8"))
print(json.dumps(response_llama, indent=2))

{
  "generation": "```json\n{\n  \"question\": \"Which player has played for both Melbourne Victory FC and Brentford F.C.?\",\n  \"logical_structure_flag\": true,\n  \"logical_structure_reasoning\": \"The question is grammatically correct and syntactically well-structured.\",\n  \"redundancy_flag\": false,\n  \"redundancy_reasoning\": \"The question does not contain its own answer.\",\n  \"multiple_answers_flag\": true,\n  \"multiple_answers_reasoning\": \"There could be multiple players who have played for both teams, making the question open to more than one valid answer.\"\n}\n```",
  "prompt_token_count": 328,
  "generation_token_count": 126,
  "stop_reason": "stop"
}


In [12]:

request_mistral = build_mistral_request_body(prompt=prompt, max_tokens=2048, temperature=0)

# Invoke the endpoint
response = bedrock_runtime.invoke_model(
    body=request_mistral["body"],
    modelId=request_mistral["modelId"],
    contentType=request_mistral["contentType"]
)

# Decode and print the response
response_mistral = json.loads(response["body"].read().decode("utf-8"))
print(json.dumps(response_mistral, indent=2))

{
  "outputs": [
    {
      "text": " {\n  \"question\": \"Which player has played for both Melbourne Victory FC and Brentford F.C.?\",\n  \"logical_structure_flag\": true,\n  \"logical_structure_reasoning\": \"The question is grammatically and syntactically correct.\",\n  \"redundancy_flag\": false,\n  \"redundancy_reasoning\": \"The question does not contain its own answer explicitly or through overly obvious phrasing.\",\n  \"multiple_answers_flag\": true,\n  \"multiple_answers_reasoning\": \"Multiple players could have played for both Melbourne Victory FC and Brentford F.C., depending on the time period considered.\"\n}",
      "stop_reason": "stop"
    }
  ]
}


In [ ]:
def invoke_bedrock_endpoint(request_body: dict) -> dict:
    """
    Invokes a Bedrock endpoint with the provided request body.

    :param request_body: The request body to send to the endpoint.
    :return: The response from the endpoint.
    """
    # Initialize the Bedrock runtime client
    bedrock_runtime = boto3.client(service_name="bedrock-runtime", region_name="us-east-1")

    # Invoke the endpoint
    response = bedrock_runtime.invoke_model(
        body=request_body["body"],
        modelId=request_body["modelId"],
        contentType=request_body["contentType"]
    )

    # Decode and return the response
    return json.loads(response["body"].read().decode("utf-8"))

In [9]:

request_mistral = build_mistral_request_body(prompt=prompt, max_tokens=2048, temperature=0)

# Invoke the endpoint
response = bedrock_runtime.invoke_model(
    body=request_mistral["body"],
    modelId=request_mistral["modelId"],
    contentType=request_mistral["contentType"]
)

# Decode and print the response
response_mistral = json.loads(response["body"].read().decode("utf-8"))
print(json.dumps(response_mistral, indent=2))

{
  "outputs": [
    {
      "text": " {\n  \"question\": \"Which player has played for both Melbourne Victory FC and Brentford F.C.?\",\n  \"logical_structure_flag\": true,\n  \"logical_structure_reasoning\": \"The question is grammatically and syntactically correct.\",\n  \"redundancy_flag\": false,\n  \"redundancy_reasoning\": \"The question does not contain its own answer explicitly or through overly obvious phrasing.\",\n  \"multiple_answers_flag\": true,\n  \"multiple_answers_reasoning\": \"Multiple players could have played for both Melbourne Victory FC and Brentford F.C., depending on the time period considered.\"\n}",
      "stop_reason": "stop"
    }
  ]
}


In [ ]:

request_llama = build_llama_request_body(prompt=prompt, max_tokens=2048, temperature=0)

# Invoke the endpoint
response = bedrock_runtime.invoke_model(
    body=request_llama["body"],
    modelId=request_llama["modelId"],
    contentType=request_llama["contentType"]
)

# Decode and print the response
response_llama = json.loads(response["body"].read().decode("utf-8"))
print(json.dumps(response_llama, indent=2))

In [66]:
json.loads(response_llama['generation'].replace('```json\n', '').replace('```', ''))

{'question': 'Which player has played for both Melbourne Victory FC and Brentford F.C.?',
 'logical_structure_flag': True,
 'logical_structure_reasoning': 'The question is grammatically correct and syntactically well-structured.',
 'redundancy_flag': False,
 'redundancy_reasoning': 'The question does not contain its own answer.',
 'multiple_answers_flag': True,
 'multiple_answers_reasoning': 'There could be multiple players who have played for both teams, making the question open to more than one valid answer.'}

In [57]:
response_commandr['chat_history'][1]['message']

'```json\n{\n  "question": "Which player has played for both Melbourne Victory FC and Brentford F.C.?",\n  "logical_structure_flag": true,\n  "logical_structure_reasoning": "The question is grammatically and syntactically correct.",\n  "redundancy_flag": false,\n  "redundancy_reasoning": "The question does not contain any explicit answers or obvious phrasing that might lead to confusion.",\n  "multiple_answers_flag": true,\n  "multiple_answers_reasoning": "The question could potentially have multiple valid answers, as more than one player might fit the description."\n}\n```'

In [14]:
request_json = build_anthropic_request_body(
            system_prompt="You are helpful assistant.",
            user_prompt=prompt,
            max_tokens=1024,
            temperature=0
        )


model_id = "us.anthropic.claude-3-5-sonnet-20241022-v2:0"

        # Invoke the endpoint.
response_data = invoke_bedrock_endpoint(request_json, model_id)
print("Response from Claude:", json.dumps(response_data, indent=2))

Response from Claude: {
  "id": "msg_bdrk_01JEUqacBNig99DAArMK2bEm",
  "type": "message",
  "role": "assistant",
  "model": "claude-3-5-sonnet-20241022",
  "content": [
    {
      "type": "text",
      "text": "{\n    \"question\": \"Which player has played for both Melbourne Victory FC and Brentford F.C.?\",\n    \"logical_structure_flag\": true,\n    \"logical_structure_reasoning\": \"The question is grammatically correct and follows proper syntax. It clearly asks about a player who has been a member of two specific football clubs.\",\n    \"redundancy_flag\": false,\n    \"redundancy_reasoning\": \"The question does not contain its own answer and requires external knowledge to respond.\",\n    \"multiple_answers_flag\": true,\n    \"multiple_answers_reasoning\": \"Over the history of both clubs, there could potentially be multiple players who have played for both teams. Without specifying a time period or additional context, more than one player might qualify as a correct answer.\"

In [21]:
request_llama = build_llama_request_body(prompt=prompt, max_tokens=2048, temperature=0)

# Invoke the endpoint
response = invoke_bedrock_endpoint(
    request_body=request_llama["body"],
    model_id=request_llama["modelId"],
    contentType=request_llama["contentType"]
)
print(response)

In [23]:
request_mistral = build_mistral_request_body(prompt=prompt, max_tokens=2048, temperature=0)

# Invoke the endpoint
response = invoke_bedrock_endpoint(
    request_body=request_mistral["body"],
    model_id=request_mistral["modelId"],
    contentType=request_mistral["contentType"]
)
print(response)

{'outputs': [{'text': ' {\n  "question": "Which player has played for both Melbourne Victory FC and Brentford F.C.?",\n  "logical_structure_flag": true,\n  "logical_structure_reasoning": "The question is grammatically and syntactically correct.",\n  "redundancy_flag": false,\n  "redundancy_reasoning": "The question does not contain its own answer explicitly or through overly obvious phrasing.",\n  "multiple_answers_flag": true,\n  "multiple_answers_reasoning": "Multiple players could have played for both Melbourne Victory FC and Brentford F.C., depending on the time period considered."\n}', 'stop_reason': 'stop'}]}


In [24]:
request_command_r = build_command_r_request_body(prompt=prompt, max_tokens=2048, temperature=0)

# Invoke the endpoint
response = invoke_bedrock_endpoint(
    request_body=request_command_r["body"],
    model_id=request_command_r["modelId"],
    contentType=request_command_r["contentType"]
)
print(response)

{'response_id': '379ed018/f31e565e-ad4e-421d-b5a8-6dea809e5f95', 'text': '```json\n{\n  "question": "Which player has played for both Melbourne Victory FC and Brentford F.C.?",\n  "logical_structure_flag": true,\n  "logical_structure_reasoning": "The question is grammatically and syntactically correct.",\n  "redundancy_flag": false,\n  "redundancy_reasoning": "The question does not contain any explicit answers or obvious phrasing that might lead to confusion.",\n  "multiple_answers_flag": true,\n  "multiple_answers_reasoning": "The question could potentially have multiple valid answers, as more than one player might fit the description."\n}\n```', 'generation_id': '513a556f-98cc-4fc7-a368-0c7307d71969', 'chat_history': [{'role': 'USER', 'message': '\nAs an expert evaluator, your role is to assess the quality and validity of trivia or natural questions. These questions aim to test the responder\'s knowledge, which may require implicit or external information. Your goal is to analyze the

In [20]:

import boto3
import logging
import time

from botocore.exceptions import ClientError

def invoke_bedrock_endpoint(
    request_body: dict,
    model_id: str,
    region_name: str = "us-east-1",
    contentType = 'application/json',
    max_retries: int = 3,
    backoff_factor: float = 2.0
) -> dict:
    """
    Invokes the Bedrock endpoint with a given request body and model ID,
    with exponential backoff retries for transient errors.

    :param request_body: JSON payload specific to the chosen model.
    :param model_id: The Bedrock model ID, e.g. "anthropic.claude-v1".
    :param region_name: The AWS region to call. Default is "us-east-1".
    :param max_retries: Number of retry attempts for transient errors.
    :param backoff_factor: Factor for exponential backoff, e.g. 2.0 means
                           1s, 2s, 4s between retries, etc.
    :return: The deserialized JSON response from Bedrock.
    """
    bedrock_runtime = boto3.client(
        service_name='bedrock-runtime',
        region_name=region_name
    )

    for attempt in range(max_retries):
        try:
            # Serialize the request body as JSON.
            # body_json = json.dumps(request_body)

            response = bedrock_runtime.invoke_model(
                body=request_body,
                modelId=model_id,
                contentType=contentType
            )

            # The response body is a StreamingBody, so we need to read and decode it.
            response_body = json.loads(response.get('body').read())
            return response_body

        except ClientError as err:
            logger.error(
                "Error invoking Bedrock on attempt %s: %s",
                attempt + 1,
                err.response["Error"]["Message"]
            )

            # If this was the last attempt, re-raise the error.
            if attempt == max_retries - 1:
                raise

            # Otherwise, back off exponentially before retrying.
            sleep_time = backoff_factor ** attempt
            logger.info(f"Retrying in {sleep_time} seconds...")
            time.sleep(sleep_time)


In [21]:
import boto3
import logging
import time

from botocore.exceptions import ClientError

logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO)
from botocore.exceptions import ClientError
def invoke_bedrock_endpoint(
    request_body: dict,
    model_id: str,
    region_name: str = "us-east-1",
    max_retries: int = 3,
    backoff_factor: float = 2.0
) -> dict:
    """
    Invokes the Bedrock endpoint with a given request body and model ID,
    with exponential backoff retries for transient errors.

    :param request_body: JSON payload specific to the chosen model.
    :param model_id: The Bedrock model ID, e.g. "anthropic.claude-v1".
    :param region_name: The AWS region to call. Default is "us-east-1".
    :param max_retries: Number of retry attempts for transient errors.
    :param backoff_factor: Factor for exponential backoff, e.g. 2.0 means
                           1s, 2s, 4s between retries, etc.
    :return: The deserialized JSON response from Bedrock.
    """
    bedrock_runtime = boto3.client(
        service_name='bedrock-runtime',
        region_name=region_name
    )

    for attempt in range(max_retries):
        try:
            # Serialize the request body as JSON.
            body_json = json.dumps(request_body)

            response = bedrock_runtime.invoke_model(
                body=body_json,
                modelId=model_id,
                contentType='application/json'
            )

            # The response body is a StreamingBody, so we need to read and decode it.
            response_body = json.loads(response.get('body').read())
            return response_body

        except ClientError as err:
            logger.error(
                "Error invoking Bedrock on attempt %s: %s",
                attempt + 1,
                err.response["Error"]["Message"]
            )

            # If this was the last attempt, re-raise the error.
            if attempt == max_retries - 1:
                raise

            # Otherwise, back off exponentially before retrying.
            sleep_time = backoff_factor ** attempt
            logger.info(f"Retrying in {sleep_time} seconds...")
            time.sleep(sleep_time)


In [10]:
request_json

{'modelId': 'mistral.mistral-small-2402-v1:0',
 'contentType': 'application/json',
 'accept': 'application/json',
 'body': '{"prompt": "<s>[INST] You are an AI assistant tasked with generating question-answer pairs from knowledge graph triples. Your goal is to create natural, human-like questions and their corresponding answers based on the provided graph data.\\n\\nTask Overview:\\nGenerate **multi-hop, complex Q&A pairs** where the questions appear simple and natural but require reasoning across multiple connected relationships within the graph to infer the answer.\\n\\nGuidelines for Generating Q&A Pairs:\\n1. **Question Design**:\\n- Questions should utilize multiple connected relationships in the graph, requiring multi-hop reasoning.\\n- Avoid single-hop or trivial questions directly derived from a single triple.\\n- The answer should be an entity or node in the graph.\\n\\n2. **Multi-Hop Reasoning**:\\n- Use paths connecting entities indirectly through multiple relationships to i

In [6]:
anthropic_req = read_jsonl_file('/home/ec2-user/preetam_experiments/res/llm_requests_test.jsonl')

In [8]:
anthropic_req[0]['modelInput']

{'anthropic_version': 'bedrock-2023-05-31',
 'system': 'You are helpful assistant.',
 'messages': [{'role': 'user',
   'content': "You are an AI assistant tasked with generating question-answer pairs from knowledge graph triples. Your goal is to create natural, human-like questions and their corresponding answers based on the provided graph data.\n\nTask Overview:\nGenerate **multi-hop, complex Q&A pairs** where the questions appear simple and natural but require reasoning across multiple connected relationships within the graph to infer the answer.\n\nGuidelines for Generating Q&A Pairs:\n1. **Question Design**:\n- Questions should utilize multiple connected relationships in the graph, requiring multi-hop reasoning.\n- Avoid single-hop or trivial questions directly derived from a single triple.\n- The answer should be an entity or node in the graph.\n\n2. **Multi-Hop Reasoning**:\n- Use paths connecting entities indirectly through multiple relationships to infer answers.\n- Questions 

In [9]:
res_file = read_jsonl_file(f"llm_requests_{split}.jsonl.out")

In [12]:
res_file[0]['modelOutput']

{'id': 'msg_bdrk_01V58UM7i7VXVVnFiiUH4fNS',
 'type': 'message',
 'role': 'assistant',
 'model': 'claude-3-5-sonnet-20241022',
 'content': [{'type': 'text',
   'text': '{\n    "valid_qa_pairs": true,\n    "number_of_qa_pairs": 3,\n    "qa_pairs": [\n        {\n            "question": "What family of animals includes both the guanaco and llama?",\n            "answer": "Camelidae",\n            "supporting_path": [\n                {\n                    "subject": "Guanaco",\n                    "predicate": "parentTaxon",\n                    "object": "Lama__u0028_genus_u0029_"\n                },\n                {\n                    "subject": "Lama__u0028_genus_u0029_",\n                    "predicate": "parentTaxon",\n                    "object": "Camelidae"\n                },\n                {\n                    "subject": "Llama",\n                    "predicate": "parentTaxon",\n                    "object": "Lama__u0028_genus_u0029_"\n                }\n            ]\n 

In [50]:
df.iloc[99]['subgraph_Steiner_largest_connected']

[['http://yago-knowledge.org/resource/Mazda_B_series',
  'http://schema.org/manufacturer',
  'http://yago-knowledge.org/resource/Mazda'],
 ['http://yago-knowledge.org/resource/Troller_T4',
  'http://schema.org/manufacturer',
  'http://yago-knowledge.org/resource/Troller_Veículos_Especiais'],
 ['http://yago-knowledge.org/resource/Ford_Ranger',
  'http://schema.org/manufacturer',
  'http://yago-knowledge.org/resource/Ford_Motor_Company'],
 ['http://yago-knowledge.org/resource/Ford_Transit',
  'http://schema.org/manufacturer',
  'http://yago-knowledge.org/resource/Ford_Motor_Company'],
 ['http://yago-knowledge.org/resource/Maurice_Jordan_Q3300959',
  'http://schema.org/worksFor',
  'http://yago-knowledge.org/resource/PSA_Group'],
 ['http://yago-knowledge.org/resource/Maurice_Jordan_Q3300959',
  'http://schema.org/worksFor',
  'http://yago-knowledge.org/resource/Peugeot'],
 ['http://yago-knowledge.org/resource/1979_World_Rally_Championship_For_Manufacturers_Q20202921',
  'http://yago-knowl

In [47]:

resp = json.loads(res_file[99]['modelOutput']['content'][0]['text'])
resp

{'valid_qa_pairs': True,
 'number_of_qa_pairs': 3,
 'qa_pairs': [{'question': 'Which car manufacturer, who owns a company that makes the T4, participated in the 1979 World Rally Championship?',
   'answer': 'Ford_Motor_Company',
   'supporting_path': [{'subject': 'Troller_T4',
     'predicate': 'manufacturer',
     'object': 'Troller_Veículos_Especiais'},
    {'subject': 'Troller_Veículos_Especiais',
     'predicate': 'ownedBy',
     'object': 'Ford_Motor_Company'},
    {'subject': '1979_World_Rally_Championship_For_Manufacturers_Q20202921',
     'predicate': 'participant',
     'object': 'Ford_Motor_Company'}]},
  {'question': 'Which company that Maurice Jordan worked for was also a participant in the 1979 World Rally Championship?',
   'answer': 'Peugeot',
   'supporting_path': [{'subject': 'Maurice_Jordan_Q3300959',
     'predicate': 'worksFor',
     'object': 'Peugeot'},
    {'subject': '1979_World_Rally_Championship_For_Manufacturers_Q20202921',
     'predicate': 'participant',
  

In [55]:
print(resp['qa_pairs'][0]['question'])
print(resp['qa_pairs'][0]['answer'])
resp['qa_pairs'][0]['supporting_path']
restore_full_triples_universal(resp['qa_pairs'][0]['supporting_path'], df.iloc[99]['subgraph_Steiner_largest_connected'])
graph = crea

Which car manufacturer, who owns a company that makes the T4, participated in the 1979 World Rally Championship?
Ford_Motor_Company


([['http://yago-knowledge.org/resource/Troller_T4',
   'http://schema.org/manufacturer',
   'http://yago-knowledge.org/resource/Troller_Veículos_Especiais'],
  ['http://yago-knowledge.org/resource/Troller_Veículos_Especiais',
   'http://yago-knowledge.org/resource/ownedBy',
   'http://yago-knowledge.org/resource/Ford_Motor_Company'],
  ['http://yago-knowledge.org/resource/1979_World_Rally_Championship_For_Manufacturers_Q20202921',
   'http://yago-knowledge.org/resource/participant',
   'http://yago-knowledge.org/resource/Ford_Motor_Company']],
 True)

In [18]:
llm_requests_list[0]['modelInput']

{'anthropic_version': 'bedrock-2023-05-31',
 'system': 'You are helpful assistant.',
 'messages': [{'role': 'user',
   'content': "You are an AI assistant tasked with generating question-answer pairs from knowledge graph triples. Your goal is to create natural, human-like questions and their corresponding answers based on the provided graph data.\n\nTask Overview:\nGenerate **multi-hop, complex Q&A pairs** where the questions appear simple and natural but require reasoning across multiple connected relationships within the graph to infer the answer.\n\nGuidelines for Generating Q&A Pairs:\n1. **Question Design**:\n- Questions should utilize multiple connected relationships in the graph, requiring multi-hop reasoning.\n- Avoid single-hop or trivial questions directly derived from a single triple.\n- The answer should be an entity or node in the graph.\n\n2. **Multi-Hop Reasoning**:\n- Use paths connecting entities indirectly through multiple relationships to infer answers.\n- Questions 

In [6]:
llm_requests_list[0]['modelInput']

{'anthropic_version': 'bedrock-2023-05-31',
 'system': 'You are helpful assistant.',
 'messages': [{'role': 'user',
   'content': "You are an AI assistant tasked with generating question-answer pairs from knowledge graph triples. Your goal is to create natural, human-like questions and their corresponding answers based on the provided graph data.\n\nTask Overview:\nGenerate **multi-hop, complex Q&A pairs** where the questions appear simple and natural but require reasoning across multiple connected relationships within the graph to infer the answer.\n\nGuidelines for Generating Q&A Pairs:\n1. **Question Design**:\n- Questions should utilize multiple connected relationships in the graph, requiring multi-hop reasoning.\n- Avoid single-hop or trivial questions directly derived from a single triple.\n- The answer should be an entity or node in the graph.\n\n2. **Multi-Hop Reasoning**:\n- Use paths connecting entities indirectly through multiple relationships to infer answers.\n- Questions 

In [19]:

model_id = "us.anthropic.claude-3-5-sonnet-20241022-v2:0"
response_data = invoke_bedrock_endpoint(llm_requests_list[0]['modelInput'], model_id)
print("Response from Claude:", json.dumps(response_data, indent=2))

INFO:botocore.credentials:Found credentials in shared credentials file: ~/.aws/credentials


Response from Claude: {
  "id": "msg_bdrk_0135t5PoReNfhHVQYXTz6SDQ",
  "type": "message",
  "role": "assistant",
  "model": "claude-3-5-sonnet-20241022",
  "content": [
    {
      "type": "text",
      "text": "{\n    \"valid_qa_pairs\": true,\n    \"number_of_qa_pairs\": 3,\n    \"qa_pairs\": [\n        {\n            \"question\": \"What family of animals includes both the alpaca and guanaco, despite them belonging to different genera?\",\n            \"answer\": \"Camelidae\",\n            \"supporting_path\": [\n                {\n                    \"subject\": \"Alpaca\",\n                    \"predicate\": \"parentTaxon\",\n                    \"object\": \"Vicugna\"\n                },\n                {\n                    \"subject\": \"Vicugna\",\n                    \"predicate\": \"parentTaxon\",\n                    \"object\": \"Camelidae\"\n                },\n                {\n                    \"subject\": \"Guanaco\",\n                    \"predicate\": \"paren

In [28]:
resp = json.loads(response_data['content'][0]['text'])
restore_full_triples_universal(resp['qa_pairs'][0]['supporting_path'], df.iloc[0]['subgraph_Steiner_largest_connected'])

([['http://yago-knowledge.org/resource/Alpaca',
   'http://schema.org/parentTaxon',
   'http://yago-knowledge.org/resource/Vicugna'],
  ['http://yago-knowledge.org/resource/Vicugna',
   'http://schema.org/parentTaxon',
   'http://yago-knowledge.org/resource/Camelidae'],
  ['http://yago-knowledge.org/resource/Guanaco',
   'http://schema.org/parentTaxon',
   'http://yago-knowledge.org/resource/Lama__u0028_genus_u0029_'],
  ['http://yago-knowledge.org/resource/Lama__u0028_genus_u0029_',
   'http://schema.org/parentTaxon',
   'http://yago-knowledge.org/resource/Camelidae']],
 True)

In [27]:
json.loads(response_data['content'][0]['text'])

{'valid_qa_pairs': True,
 'number_of_qa_pairs': 3,
 'qa_pairs': [{'question': 'What family of animals includes both the alpaca and guanaco, despite them belonging to different genera?',
   'answer': 'Camelidae',
   'supporting_path': [{'subject': 'Alpaca',
     'predicate': 'parentTaxon',
     'object': 'Vicugna'},
    {'subject': 'Vicugna', 'predicate': 'parentTaxon', 'object': 'Camelidae'},
    {'subject': 'Guanaco',
     'predicate': 'parentTaxon',
     'object': 'Lama__u0028_genus_u0029_'},
    {'subject': 'Lama__u0028_genus_u0029_',
     'predicate': 'parentTaxon',
     'object': 'Camelidae'}]},
  {'question': 'Which higher-level taxonomic group connects both the Chevrotain and the Pecora through their evolutionary relationships?',
   'answer': 'Ruminant',
   'supporting_path': [{'subject': 'Chevrotain',
     'predicate': 'parentTaxon',
     'object': 'Tragulina'},
    {'subject': 'Tragulina', 'predicate': 'parentTaxon', 'object': 'Ruminant'},
    {'subject': 'Pecora', 'predicate'

In [ ]:
request_json = build_anthropic_request_body(
            system_prompt="You are helpful assistant.",
            user_prompt=prompt,
            max_tokens=1024,
            temperature=0
        )


model_id = "us.anthropic.claude-3-5-sonnet-20241022-v2:0"

        # Invoke the endpoint.
response_data = invoke_bedrock_endpoint(request_json, model_id)
print("Response from Claude:", json.dumps(response_data, indent=2))

In [8]:
resp = json.loads(response_data['content'][0]['text'])

In [1]:
resp

NameError: name 'resp' is not defined

In [13]:
lenjson.dumps(request_json)

'{"anthropic_version": "bedrock-2023-05-31", "system": "You are helpful assistant.", "messages": [{"role": "user", "content": "\\nYou are an AI assistant tasked with generating question-answer pairs from knowledge graph triples. Your goal is to create natural, human-like questions and their corresponding answers based on the provided graph data.\\n\\nTask Overview:\\nGenerate **multi-hop, complex Q&A pairs** where the questions appear simple and natural but require reasoning across multiple connected relationships within the graph to infer the answer.\\n\\nGuidelines for Generating Q&A Pairs:\\n1. **Question Design**:\\n- Questions should utilize multiple connected relationships in the graph, requiring multi-hop reasoning.\\n- Avoid single-hop or trivial questions directly derived from a single triple.\\n- The answer should be an entity or node in the graph.\\n\\n2. **Multi-Hop Reasoning**:\\n- Use paths connecting entities indirectly through multiple relationships to infer answers.\\n

In [9]:
restore_full_triples_universal(resp['qa_pairs'][0]['supporting_path'], df.iloc[idx]['subgraph_Steiner_largest_connected'])

([['http://yago-knowledge.org/resource/Alpaca',
   'http://schema.org/parentTaxon',
   'http://yago-knowledge.org/resource/Vicugna'],
  ['http://yago-knowledge.org/resource/Vicugna',
   'http://schema.org/parentTaxon',
   'http://yago-knowledge.org/resource/Camelidae'],
  ['http://yago-knowledge.org/resource/Guanaco',
   'http://schema.org/parentTaxon',
   'http://yago-knowledge.org/resource/Lama__u0028_genus_u0029_'],
  ['http://yago-knowledge.org/resource/Lama__u0028_genus_u0029_',
   'http://schema.org/parentTaxon',
   'http://yago-knowledge.org/resource/Camelidae']],
 True)

In [25]:
df.iloc[idx]['subgraph_Steiner_largest_connected']

[['http://yago-knowledge.org/resource/Camelidae',
  'http://schema.org/parentTaxon',
  'http://yago-knowledge.org/resource/Even-toed_ungulate'],
 ['http://yago-knowledge.org/resource/Even-toed_ungulate',
  'http://schema.org/parentTaxon',
  'http://yago-knowledge.org/resource/Ungulate'],
 ['http://yago-knowledge.org/resource/Tragulina',
  'http://schema.org/parentTaxon',
  'http://yago-knowledge.org/resource/Ruminant'],
 ['http://yago-knowledge.org/resource/Chevrotain',
  'http://schema.org/parentTaxon',
  'http://yago-knowledge.org/resource/Even-toed_ungulate'],
 ['http://yago-knowledge.org/resource/Chevrotain',
  'http://schema.org/parentTaxon',
  'http://yago-knowledge.org/resource/Tragulina'],
 ['http://yago-knowledge.org/resource/Camel',
  'http://schema.org/parentTaxon',
  'http://yago-knowledge.org/resource/Camelidae'],
 ['http://yago-knowledge.org/resource/Guanaco',
  'http://schema.org/parentTaxon',
  'http://yago-knowledge.org/resource/Lama__u0028_genus_u0029_'],
 ['http://ya

In [26]:
resp['qa_pairs'][0]['supporting_path']

[{'subject': 'Vicuña', 'predicate': 'parentTaxon', 'object': 'Vicugna'},
 {'subject': 'Vicugna', 'predicate': 'parentTaxon', 'object': 'Camelidae'},
 {'subject': 'Guanaco',
  'predicate': 'parentTaxon',
  'object': 'Lama__u0028_genus_u0029_'},
 {'subject': 'Lama__u0028_genus_u0029_',
  'predicate': 'parentTaxon',
  'object': 'Camelidae'}]